<a href="https://colab.research.google.com/github/DistrictRobotics/3d-resources/blob/main/DREV_Fleet_2026_AI_Asset_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys, subprocess
print(sys.version)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)
import torch
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'available:', torch.cuda.is_available())

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Torch: 2.14.1+cu130 CUDA: 13.0 available: True


In [ ]:
import os, pathlib, subprocess, sys
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
os.environ['MAX_JOBS'] = '2'
repo = pathlib.Path('/content/TripoSR')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/VAST-AI-Research/TripoSR.git',str(repo)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','omegaconf','einops','transformers<5','trimesh','rembg','onnxruntime','huggingface-hub','imageio[ffmpeg]','xatlas','moderngl','ninja','cmake'],check=True)
subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','git+https://github.com/tatsy/torchmcubes.git'],check=True)
print('DREV model runtime dependencies installed')

CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '--no-build-isolation', 'git+https://github.com/tatsy/torchmcubes.git']' returned non-zero exit status 2.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','scikit-build-core','pybind11'],check=True)
r = subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','git+https://github.com/tatsy/torchmcubes.git'],capture_output=True,text=True)
print(r.stdout[-6000:])
print(r.stderr[-6000:])
print('Exit:',r.returncode)
assert r.returncode == 0

ocal/lib/python3.13/dist-packages (from torch==2.14.*->torchmcubes==0.1.0) (80.10.2)
  Created wheel for torchmcubes: filename=torchmcubes-0.1.0-cp313-cp313-linux_x86_64.whl size=169315 sha256=83522bbfac7e707e48d6452d1cd37d6e6c00469433bb99f244abdb45a525d40c
  Stored in directory: /tmp/pip-ephem-wheel-cache-m_7wxdhd/wheels/f5/37/ea/f954b2bf2c314b49db188fa848c8eee1270e2c56826652af42
Successfully built torchmcubes

  Running command git clone --filter=blob:none --quiet https://github.com/tatsy/torchmcubes.git /tmp/pip-req-build-mbrtrksk

Exit: 0


In [ ]:
import zipfile
with zipfile.ZipFile('/content/DREV-2026-reference-inputs.zip') as z:
    print(z.namelist())
    z.extractall('/content/drev-references')
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
import torchmcubes
print('Mesh extraction dependency ready')
print(subprocess.run(['git','-C','/content/TripoSR','rev-parse','HEAD'],capture_output=True,text=True).stdout)
print(pathlib.Path('/content/TripoSR/tsr/models/isosurface.py').read_text()[:10000])

['2026-cybertruck.png', '2026-gmc-hummer-ev-suv.png', '2026-lucid-air.png', '2026-mercedes-eqs.png', '2026-mustang-mach-e.png']
name, memory.total [MiB]
Tesla T4, 15360 MiB

Mesh extraction dependency ready
107cefdc244c39106fa830359024f6a2f1c78871

from typing import Callable, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torchmcubes import marching_cubes


class IsosurfaceHelper(nn.Module):
    points_range: Tuple[float, float] = (0, 1)

    @property
    def grid_vertices(self) -> torch.FloatTensor:
        raise NotImplementedError


class MarchingCubeHelper(IsosurfaceHelper):
    def __init__(self, resolution: int) -> None:
        super().__init__()
        self.resolution = resolution
        self.mc_func: Callable = marching_cubes
        self._grid_vertices: Optional[torch.FloatTensor] = None

    @property
    def grid_vertices(self) -> torch.FloatTensor:
        if self._grid_vertices is None:
            # keep the vertices on CPU so that we can

In [ ]:
os.environ['PYOPENGL_PLATFORM']='egl'
asset='2026-cybertruck'
cmd=[sys.executable,'run.py',f'/content/drev-references/{asset}.png','--output-dir',f'/content/drev-generated/{asset}','--model-save-format','obj','--bake-texture','--texture-resolution','2048','--mc-resolution','256','--chunk-size','4096']
log_path=pathlib.Path('/content/cybertruck-generation.log')
with log_path.open('w') as log:
    result=subprocess.run(cmd,cwd='/content/TripoSR',stdout=log,stderr=subprocess.STDOUT,text=True)
print(log_path.read_text()[-14000:])
print('Exit:',result.returncode)
assert result.returncode==0

      | 239M/1.02G [00:08<00:29, 26.9MB/s]
 24%|█████████                             | 243M/1.02G [00:08<00:27, 28.8MB/s]
 24%|█████████▏                            | 246M/1.02G [00:08<00:25, 30.7MB/s]
 24%|█████████▎                            | 250M/1.02G [00:08<00:23, 32.4MB/s]
 25%|█████████▍                            | 254M/1.02G [00:08<00:22, 33.6MB/s]
 25%|█████████▌                            | 258M/1.02G [00:08<00:21, 35.0MB/s]
 26%|█████████▋                            | 262M/1.02G [00:09<00:20, 36.8MB/s]
 26%|█████████▊                            | 266M/1.02G [00:09<00:19, 38.5MB/s]
 26%|██████████                            | 270M/1.02G [00:09<00:18, 39.9MB/s]
 27%|██████████▏                           | 275M/1.02G [00:09<00:17, 41.6MB/s]
 27%|██████████▎                           | 280M/1.02G [00:09<00:17, 43.2MB/s]
 28%|██████████▌                           | 284M/1.02G [00:09<00:16, 44.5MB/s]
 28%|██████████▋                           | 289M/1.02G [00:09<00:16, 44.9MB/

AssertionError: 

In [ ]:
bake_path=pathlib.Path('/content/TripoSR/tsr/bake_texture.py')
bake_source=bake_path.read_text()
bake_path.write_text(bake_source.replace('moderngl.create_context(standalone=True)', 'moderngl.create_context(standalone=True, backend="egl")'))
with log_path.open('w') as log:
    result=subprocess.run(cmd,cwd='/content/TripoSR',stdout=log,stderr=subprocess.STDOUT,text=True)
print(log_path.read_text()[-5000:])
print('Exit:',result.returncode)
assert result.returncode==0

2026-11-17 04:58:34,288 - INFO - Initializing model ...
2026-11-17 04:58:37,719 - INFO - NumExpr defaulting to 2 threads.
I0000 00:00:1794891518.778785    4542 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794891522.822776    4542 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-17 04:58:56,456 - INFO - Initializing model finished in 22167.99ms.
2026-11-17 04:58:56,456 - INFO - Processing images ...
2026-11-17 04:59:01,094 - INFO - Processing images finished in 4637.51ms.
2026-11-17 04:59:01,094 - INFO - Running image 1/1 ...
2026-11-17 04:59:01,094 - INFO - Running model ...
2026-11-17 04:59:02,955 - INFO - Running model finished in 1860.62ms.
2026-11-17 04:59:02,955 - INFO - Extracting mesh ...
2026-11-17 04:59:

AssertionError: 

In [ ]:
import trimesh, json
from PIL import Image
mesh_dir=pathlib.Path('/content/drev-generated/2026-cybertruck/0')
print('Output files:',[p.name for p in mesh_dir.iterdir()])
assert (mesh_dir/'mesh.obj').exists() and (mesh_dir/'texture.png').exists()
mesh=trimesh.load(mesh_dir/'mesh.obj',force='mesh',process=False)
assert mesh.visual.uv is not None
material=trimesh.visual.material.PBRMaterial(baseColorTexture=Image.open(mesh_dir/'texture.png'),metallicFactor=0.35,roughnessFactor=0.45)
mesh.visual=trimesh.visual.TextureVisuals(uv=mesh.visual.uv,material=material)
mesh.metadata.update({'generator':'TripoSR','reference':'DREV AI reference 2026-cybertruck.png','accepted':False})
out=pathlib.Path('/content/drev-generated/2026-cybertruck.glb')
mesh.export(out)
print(json.dumps({'file':str(out),'bytes':out.stat().st_size,'triangles':len(mesh.faces),'vertices':len(mesh.vertices),'bounds':mesh.bounds.tolist()}))

Output files: ['texture.png', 'mesh.obj', 'input.png']
{"file": "/content/drev-generated/2026-cybertruck.glb", "bytes": 4780908, "triangles": 85872, "vertices": 66936, "bounds": [[-0.433179, -0.44404, -0.187305], [0.440771, 0.514478, 0.17999]]}


In [ ]:
from google.colab import files
files.download('/content/drev-generated/2026-cybertruck.glb')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import base64,zlib,hashlib
from IPython.display import HTML,display
payload=pathlib.Path('/content/drev-generated/2026-cybertruck.glb').read_bytes()
encoded=base64.b64encode(zlib.compress(payload,9)).decode('ascii')
print({'bytes':len(payload),'encodedBytes':len(encoded),'sha256':hashlib.sha256(payload).hexdigest()})
display(HTML('<label>Generated Cybertruck GLB transfer</label><textarea aria-label="Generated Cybertruck GLB transfer" style="width:100%;height:100px" readonly>'+encoded+'</textarea>'))

{'bytes': 4780908, 'encodedBytes': 4804272, 'sha256': '9ac3408653c5aa43c502a6f4b9f099af156ef16b53129f1e221d4069099b9cfd'}


In [ ]:
import os,sys,subprocess,pathlib,json,hashlib,time,zipfile,shutil
REPO=pathlib.Path('/content/TripoSR')
# @title 2. Choose storage and load the fleet bundle
USE_GOOGLE_DRIVE = False # @param {type:"boolean"}
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=pathlib.Path('/content/drive/MyDrive/DREV Fleet 2026')
else:
    BASE=pathlib.Path('/content/DREV Fleet 2026')
    print('Temporary session storage: download checkpoints before disconnecting.')
BASE.mkdir(parents=True,exist_ok=True)
INPUTS=BASE/'inputs'; OUTPUTS=BASE/'outputs'
INPUTS.mkdir(exist_ok=True); OUTPUTS.mkdir(exist_ok=True)
BUNDLE=pathlib.Path('/content/DREV-Fleet-Colab-Bundle.zip')
if not BUNDLE.exists() and not (INPUTS/'queue.json').exists():
    from google.colab import files
    print('Upload DREV-Fleet-Colab-Bundle.zip')
    files.upload()
if BUNDLE.exists():
    with zipfile.ZipFile(BUNDLE) as z:
        for item in z.infolist():
            dest=(INPUTS/item.filename).resolve()
            if not dest.is_relative_to(INPUTS.resolve()): raise ValueError('Unsafe archive path')
        z.extractall(INPUTS)
QUEUE=json.loads((INPUTS/'queue.json').read_text())
WORKER=INPUTS/'fleet_colab_worker.py'
assert WORKER.exists()
print('Targets:',len(QUEUE),'References available:',sum((INPUTS/'references'/r['reference']).exists() for r in QUEUE))
print('Checkpoint folder:',OUTPUTS)

# @title 3. Resume geometry queue (one vehicle at a time)
MAX_VEHICLES = 95 # @param {type:"integer"}
TEXTURE_SIZE = 2048 # @param [1024, 2048] {type:"raw"}
CHUNK_SIZE = 4096 # @param [1024, 2048, 4096] {type:"raw"}
STAGE='geometry'
report=[]; attempted=0
for row in QUEUE:
    image=INPUTS/'references'/row['reference']
    item={'id':row['id'],'name':row['name'],'stage':STAGE,'accepted':False}
    if not image.exists():
        item['state']='waiting-for-reference'; report.append(item); continue
    image_hash=hashlib.sha256(image.read_bytes()).hexdigest()
    folder=OUTPUTS/row['id']/image_hash[:12]
    marker=folder/(STAGE+'.json')
    expected={'resolution':256} if STAGE=='geometry' else {'textureResolution':TEXTURE_SIZE,'chunkSize':CHUNK_SIZE}
    if marker.exists():
        saved=json.loads(marker.read_text())
        verified=saved.get('inputSha256')==image_hash and saved.get('settings')==expected and bool(saved.get('files'))
        verified=verified and all((folder/n).exists() and hashlib.sha256((folder/n).read_bytes()).hexdigest()==h for n,h in saved.get('files',{}).items())
        if verified:
            item['state']='complete';report.append(item);continue
    if STAGE=='texture' and not (folder/'geometry.json').exists():
        item['state']='waiting-for-geometry'; report.append(item);continue
    if attempted>=MAX_VEHICLES:
        item['state']='queued'; report.append(item);continue
    attempted+=1; folder.mkdir(parents=True,exist_ok=True)
    command=[sys.executable,str(WORKER),'--stage',STAGE,'--asset',row['id'],'--name',row['name'],'--image',str(image),'--output',str(OUTPUTS),'--texture-resolution',str(TEXTURE_SIZE),'--chunk-size',str(CHUNK_SIZE)]
    env=dict(os.environ,PYTHONPATH=str(REPO))
    print(f"{STAGE}: {row['name']}",flush=True)
    try:
        with (folder/(STAGE+'.log')).open('w') as log:
            result=subprocess.run(command,cwd=REPO,env=env,stdout=log,stderr=subprocess.STDOUT,timeout=1800)
        code=result.returncode
    except subprocess.TimeoutExpired:
        code=124
        item['reason']='Timed out after 30 minutes; other vehicles can continue.'
    item['state']='complete' if code==0 else 'failed'
    if code:
        print((folder/(STAGE+'.log')).read_text()[-2000:])
    report.append(item)
    # Every vehicle checkpoint is saved independently; a restart skips verified work.
    checkpoint=BASE/(STAGE+'-queue-status.json')
    temporary=checkpoint.with_suffix('.tmp');temporary.write_text(json.dumps(report,indent=2));temporary.replace(checkpoint)
    print(item['state'],flush=True)
print(json.dumps({state:sum(r['state']==state for r in report) for state in sorted({r['state'] for r in report})},indent=2))


Temporary session storage: download checkpoints before disconnecting.
Targets: 95 References available: 5
Checkpoint folder: /content/DREV Fleet 2026/outputs
geometry: Lucid Air Sapphire
complete
geometry: Ford Mustang Mach-E GT
complete
geometry: Mercedes-Benz EQS
complete
geometry: GMC Hummer EV SUV
complete
geometry: Tesla Cybertruck
complete
{
  "complete": 5,
  "waiting-for-reference": 90
}


In [ ]:
import os,sys,subprocess,pathlib,json,hashlib,time,zipfile,shutil
REPO=pathlib.Path('/content/TripoSR')
assert pathlib.Path('/content/drive/MyDrive').is_dir(), 'Drive mount not ready'
source_root=pathlib.Path('/content/DREV Fleet 2026')
drive_root=pathlib.Path('/content/drive/MyDrive/DREV Fleet 2026')
for source in source_root.rglob('*'):
    if source.is_file():
        destination=drive_root/source.relative_to(source_root)
        destination.parent.mkdir(parents=True,exist_ok=True)
        if destination.exists() and hashlib.sha256(destination.read_bytes()).digest()!=hashlib.sha256(source.read_bytes()).digest():
            raise RuntimeError('Existing Drive checkpoint differs: '+str(destination))
        if not destination.exists(): shutil.copy2(source,destination)
print('Geometry checkpoints copied to Google Drive')
# @title 2. Choose storage and load the fleet bundle
USE_GOOGLE_DRIVE = True # @param {type:"boolean"}
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=pathlib.Path('/content/drive/MyDrive/DREV Fleet 2026')
else:
    BASE=pathlib.Path('/content/DREV Fleet 2026')
    print('Temporary session storage: download checkpoints before disconnecting.')
BASE.mkdir(parents=True,exist_ok=True)
INPUTS=BASE/'inputs'; OUTPUTS=BASE/'outputs'
INPUTS.mkdir(exist_ok=True); OUTPUTS.mkdir(exist_ok=True)
BUNDLE=pathlib.Path('/content/DREV-Fleet-Colab-Bundle-v2.zip')
if not BUNDLE.exists() and not (INPUTS/'queue.json').exists():
    from google.colab import files
    print('Upload DREV-Fleet-Colab-Bundle-v2.zip')
    files.upload()
if BUNDLE.exists():
    with zipfile.ZipFile(BUNDLE) as z:
        for item in z.infolist():
            dest=(INPUTS/item.filename).resolve()
            if not dest.is_relative_to(INPUTS.resolve()): raise ValueError('Unsafe archive path')
        z.extractall(INPUTS)
QUEUE=json.loads((INPUTS/'queue.json').read_text())
WORKER=INPUTS/'fleet_colab_worker.py'
assert WORKER.exists()
print('Targets:',len(QUEUE),'References available:',sum((INPUTS/'references'/r['reference']).exists() for r in QUEUE))
print('Checkpoint folder:',OUTPUTS)

shutil.copytree(INPUTS/'notebooks',BASE/'notebooks',dirs_exist_ok=True)
shutil.copy2('/content/drev-generated/2026-cybertruck.glb',BASE/'recovered-cybertruck.glb')
# @title 3. Resume geometry queue (one vehicle at a time)
MAX_VEHICLES = 95 # @param {type:"integer"}
TEXTURE_SIZE = 2048 # @param [1024, 2048] {type:"raw"}
CHUNK_SIZE = 4096 # @param [1024, 2048, 4096] {type:"raw"}
STAGE='geometry'
report=[]; attempted=0
for row in QUEUE:
    image=INPUTS/'references'/row['reference']
    item={'id':row['id'],'name':row['name'],'stage':STAGE,'accepted':False}
    if not image.exists():
        item['state']='waiting-for-reference'; report.append(item); continue
    image_hash=hashlib.sha256(image.read_bytes()).hexdigest()
    folder=OUTPUTS/row['id']/image_hash[:12]
    marker=folder/(STAGE+'.json')
    expected={'resolution':256} if STAGE=='geometry' else {'textureResolution':TEXTURE_SIZE,'chunkSize':CHUNK_SIZE}
    if marker.exists():
        saved=json.loads(marker.read_text())
        verified=saved.get('inputSha256')==image_hash and saved.get('settings')==expected and bool(saved.get('files'))
        verified=verified and all((folder/n).exists() and hashlib.sha256((folder/n).read_bytes()).hexdigest()==h for n,h in saved.get('files',{}).items())
        if verified:
            item['state']='complete';report.append(item);continue
    if STAGE=='texture' and not (folder/'geometry.json').exists():
        item['state']='waiting-for-geometry'; report.append(item);continue
    if attempted>=MAX_VEHICLES:
        item['state']='queued'; report.append(item);continue
    attempted+=1; folder.mkdir(parents=True,exist_ok=True)
    command=[sys.executable,str(WORKER),'--stage',STAGE,'--asset',row['id'],'--name',row['name'],'--image',str(image),'--output',str(OUTPUTS),'--texture-resolution',str(TEXTURE_SIZE),'--chunk-size',str(CHUNK_SIZE)]
    env=dict(os.environ,PYTHONPATH=str(REPO))
    print(f"{STAGE}: {row['name']}",flush=True)
    try:
        with (folder/(STAGE+'.log')).open('w') as log:
            result=subprocess.run(command,cwd=REPO,env=env,stdout=log,stderr=subprocess.STDOUT,timeout=1800)
        code=result.returncode
    except subprocess.TimeoutExpired:
        code=124
        item['reason']='Timed out after 30 minutes; other vehicles can continue.'
    item['state']='complete' if code==0 else 'failed'
    if code:
        print((folder/(STAGE+'.log')).read_text()[-2000:])
    report.append(item)
    # Every vehicle checkpoint is saved independently; a restart skips verified work.
    checkpoint=BASE/(STAGE+'-queue-status.json')
    temporary=checkpoint.with_suffix('.tmp');temporary.write_text(json.dumps(report,indent=2));temporary.replace(checkpoint)
    print(item['state'],flush=True)
checkpoint=BASE/(STAGE+'-queue-status.json')
temporary=checkpoint.with_suffix('.tmp');temporary.write_text(json.dumps(report,indent=2));temporary.replace(checkpoint)
print(json.dumps({state:sum(r['state']==state for r in report) for state in sorted({r['state'] for r in report})},indent=2))

# @title 3. Resume texture queue (one vehicle at a time)
MAX_VEHICLES = 95 # @param {type:"integer"}
TEXTURE_SIZE = 2048 # @param [1024, 2048] {type:"raw"}
CHUNK_SIZE = 4096 # @param [1024, 2048, 4096] {type:"raw"}
STAGE='texture'
report=[]; attempted=0
for row in QUEUE:
    image=INPUTS/'references'/row['reference']
    item={'id':row['id'],'name':row['name'],'stage':STAGE,'accepted':False}
    if not image.exists():
        item['state']='waiting-for-reference'; report.append(item); continue
    image_hash=hashlib.sha256(image.read_bytes()).hexdigest()
    folder=OUTPUTS/row['id']/image_hash[:12]
    marker=folder/(STAGE+'.json')
    expected={'resolution':256} if STAGE=='geometry' else {'textureResolution':TEXTURE_SIZE,'chunkSize':CHUNK_SIZE}
    if marker.exists():
        saved=json.loads(marker.read_text())
        verified=saved.get('inputSha256')==image_hash and saved.get('settings')==expected and bool(saved.get('files'))
        verified=verified and all((folder/n).exists() and hashlib.sha256((folder/n).read_bytes()).hexdigest()==h for n,h in saved.get('files',{}).items())
        if verified:
            item['state']='complete';report.append(item);continue
    if STAGE=='texture' and not (folder/'geometry.json').exists():
        item['state']='waiting-for-geometry'; report.append(item);continue
    if attempted>=MAX_VEHICLES:
        item['state']='queued'; report.append(item);continue
    attempted+=1; folder.mkdir(parents=True,exist_ok=True)
    command=[sys.executable,str(WORKER),'--stage',STAGE,'--asset',row['id'],'--name',row['name'],'--image',str(image),'--output',str(OUTPUTS),'--texture-resolution',str(TEXTURE_SIZE),'--chunk-size',str(CHUNK_SIZE)]
    env=dict(os.environ,PYTHONPATH=str(REPO))
    print(f"{STAGE}: {row['name']}",flush=True)
    try:
        with (folder/(STAGE+'.log')).open('w') as log:
            result=subprocess.run(command,cwd=REPO,env=env,stdout=log,stderr=subprocess.STDOUT,timeout=1800)
        code=result.returncode
    except subprocess.TimeoutExpired:
        code=124
        item['reason']='Timed out after 30 minutes; other vehicles can continue.'
    item['state']='complete' if code==0 else 'failed'
    if code:
        print((folder/(STAGE+'.log')).read_text()[-2000:])
    report.append(item)
    # Every vehicle checkpoint is saved independently; a restart skips verified work.
    checkpoint=BASE/(STAGE+'-queue-status.json')
    temporary=checkpoint.with_suffix('.tmp');temporary.write_text(json.dumps(report,indent=2));temporary.replace(checkpoint)
    print(item['state'],flush=True)
checkpoint=BASE/(STAGE+'-queue-status.json')
temporary=checkpoint.with_suffix('.tmp');temporary.write_text(json.dumps(report,indent=2));temporary.replace(checkpoint)
print(json.dumps({state:sum(r['state']==state for r in report) for state in sorted({r['state'] for r in report})},indent=2))


Geometry checkpoints copied to Google Drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Targets: 95 References available: 7
Checkpoint folder: /content/drive/MyDrive/DREV Fleet 2026/outputs
geometry: Arc Sport
complete
geometry: Trek Rail+
complete
{
  "complete": 7,
  "waiting-for-reference": 88
}
texture: Lucid Air Sapphire
complete
texture: Ford Mustang Mach-E GT
complete
texture: Mercedes-Benz EQS
complete
texture: GMC Hummer EV SUV
complete
texture: Tesla Cybertruck
complete
texture: Arc Sport
complete
texture: Trek Rail+
complete
{
  "complete": 7,
  "waiting-for-reference": 88
}


In [ ]:
import os,sys,subprocess,pathlib,json,time
from google.colab import drive
drive.mount('/content/drive')
BASE=pathlib.Path('/content/drive/MyDrive/DREV Fleet 2026')
assert (BASE/'outputs').is_dir(), 'Saved outputs missing'
QUALITY=BASE/'quality-lab';QUALITY.mkdir(exist_ok=True)
REPO=pathlib.Path('/content/Hunyuan3D-2.1')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git',str(REPO)],check=True)
print('Repository ready')
deps=['diffusers==0.30.0','accelerate==1.1.1','transformers==4.46.0','huggingface-hub==0.30.2','safetensors','omegaconf','einops','trimesh','pymeshlab','PyMCubes','torchdiffeq','rembg','onnxruntime','timm','imageio','opencv-python','pytorch-lightning==1.9.5','torchmetrics==1.6.0']
with (QUALITY/'setup.log').open('w') as log:
    setup_result=subprocess.run([sys.executable,'-m','pip','install',*deps],stdout=log,stderr=subprocess.STDOUT,text=True)
print((QUALITY/'setup.log').read_text()[-5000:])
assert setup_result.returncode==0
print('Quality engine dependencies ready')
print((REPO/'hy3dshape/requirements.txt').read_text() if (REPO/'hy3dshape/requirements.txt').exists() else 'Shape requirements use main environment')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository ready
transformers' candidate (version 4.46.0 at https://files.pythonhosted.org/packages/db/88/1ef8a624a33d7fe460a686b9e0194a7916320fc0d67d4e38e570beeac039/transformers-4.46.0-py3-none-any.whl (from https://pypi.org/simple/transformers/) (requires-python:>=3.8.0))
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 73.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 160.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 43.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.4/926.4 kB 48.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 126.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 41.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [ ]:
probe_script=QUALITY/'shape_import_probe.py'
probe_script.write_text("import sys,inspect\nsys.path.insert(0,'/content/Hunyuan3D-2.1/hy3dshape')\nfrom hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline\nprint('SHAPE IMPORT READY')\nprint(inspect.signature(Hunyuan3DDiTFlowMatchingPipeline.from_pretrained))\nprint(inspect.signature(Hunyuan3DDiTFlowMatchingPipeline.__call__))\n")
with (QUALITY/'shape-import.log').open('w') as log:
    probe_result=subprocess.run([sys.executable,str(probe_script)],cwd=REPO,stdout=log,stderr=subprocess.STDOUT,text=True)
print((QUALITY/'shape-import.log').read_text()[-6000:])
print('Exit:',probe_result.returncode)
print(subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'],capture_output=True,text=True).stdout)


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.

0it [00:00, ?it/s]
0it [00:00, ?it/s]
I0000 00:00:1794896355.427389    3965 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794896355.499100    3965 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794896358.046076    3965 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results 

In [ ]:
shape_worker=QUALITY/'generate_quality_shape.py'
shape_worker.write_text("import sys,json,time,pathlib,torch\nfrom PIL import Image\nsys.path.insert(0,'/content/Hunyuan3D-2.1/hy3dshape')\nfrom hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline\nbase=pathlib.Path('/content/drive/MyDrive/DREV Fleet 2026')\noutput=base/'quality-lab'\nimage=base/'inputs/references/2026-cybertruck.png'\nstart=time.time()\nprint('Loading Hunyuan3D-2.1 shape weights',flush=True)\npipeline=Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2.1',use_safetensors=False)\nmesh=pipeline(image=Image.open(image).convert('RGBA'),num_inference_steps=50,guidance_scale=5.0,generator=torch.Generator().manual_seed(2026),octree_resolution=512,num_chunks=8000)[0]\nmesh.metadata.update({'name':'Tesla Cybertruck','generator':'Hunyuan3D-2.1','accepted':False})\npath=output/'cybertruck-shape-hunyuan.glb'\nmesh.export(path)\nrecord={'file':str(path),'bytes':path.stat().st_size,'triangles':len(mesh.faces),'seconds':round(time.time()-start,2),'accepted':False}\n(output/'cybertruck-shape-hunyuan.json').write_text(json.dumps(record,indent=2))\nprint(json.dumps(record),flush=True)\n")
with (QUALITY/'shape-generation.log').open('w') as log:
    shape_result=subprocess.run([sys.executable,str(shape_worker)],cwd=REPO,stdout=log,stderr=subprocess.STDOUT,text=True)
print((QUALITY/'shape-generation.log').read_text()[-5000:])
print('Exit:',shape_result.returncode)


it/s]
Volume Decoding: 100%|██████████| 16876/16876 [02:17<00:00, 122.92it/s]
{"file": "/content/drive/MyDrive/DREV Fleet 2026/quality-lab/cybertruck-shape-hunyuan.glb", "bytes": 13020272, "triangles": 723344, "seconds": 221.81, "accepted": false}

Exit: 0


In [ ]:
import torch,urllib.request
os.environ['MAX_JOBS']='2'
os.environ['TORCH_CUDA_ARCH_LIST']='.'.join(map(str,torch.cuda.get_device_capability()))
paint_deps=['ninja','pybind11','configargparse','pygltflib','xatlas','fast-simplification','realesrgan==0.3.0','basicsr==1.4.2','bpy','cupy-cuda12x==13.4.1']
with (QUALITY/'paint-setup.log').open('w') as log:
    paint_setup_result=subprocess.run([sys.executable,'-m','pip','install',*paint_deps],stdout=log,stderr=subprocess.STDOUT,text=True)
print((QUALITY/'paint-setup.log').read_text()[-4000:])
assert paint_setup_result.returncode==0
with (QUALITY/'rasterizer-build.log').open('w') as log:
    raster_result=subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation',str(REPO/'hy3dpaint/custom_rasterizer')],stdout=log,stderr=subprocess.STDOUT,text=True)
print((QUALITY/'rasterizer-build.log').read_text()[-4000:])
assert raster_result.returncode==0
with (QUALITY/'inpaint-build.log').open('w') as log:
    inpaint_result=subprocess.run(['bash','compile_mesh_painter.sh'],cwd=REPO/'hy3dpaint/DifferentiableRenderer',stdout=log,stderr=subprocess.STDOUT,text=True)
print((QUALITY/'inpaint-build.log').read_text()[-2000:])
assert inpaint_result.returncode==0
checkpoint=REPO/'hy3dpaint/ckpt/RealESRGAN_x4plus.pth';checkpoint.parent.mkdir(exist_ok=True)
if not checkpoint.exists():
    urllib.request.urlretrieve('https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',checkpoint)
simplify=REPO/'hy3dpaint/utils/simplify_mesh_utils.py'
simplify.write_text(simplify.read_text().replace('simplify_quadric_decimation(target_count)','simplify_quadric_decimation(face_count=target_count)'))
(QUALITY/'installed-versions.txt').write_text(subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True).stdout)
print('Separate PBR texture dependencies built')


thon3.13/dist-packages (from scikit-image->basicsr==1.4.2) (2026.10.15)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 MB 8.6 MB/s  0:00:12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 53.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.5/427.5 MB 36.0 MB/s  0:00:09
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rembg 2.0.87 requires numpy<3.0.0,>=2.3.0, but you have numpy 2.2.6 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.30.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.

Processing ./Hunyuan3D-2.1/hy3dpaint/custom_rasterizer
  Preparing metadata (

In [17]:
import os,sys,subprocess,pathlib,textwrap
BASE=pathlib.Path('/content/drive/MyDrive/DREV Fleet 2026')
QUALITY=BASE/'quality-lab'
worker=QUALITY/'generate_quality_pbr.py'
worker.write_text(textwrap.dedent('''
import os,sys,pathlib,time,json
ROOT=pathlib.Path('/content/Hunyuan3D-2.1')
BASE=pathlib.Path('/content/drive/MyDrive/DREV Fleet 2026')
OUT=BASE/'quality-lab'
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
from torchvision_fix import apply_fix
apply_fix()
sys.path.insert(0,str(ROOT/'hy3dpaint'))
from textureGenPipeline import Hunyuan3DPaintPipeline,Hunyuan3DPaintConfig
config=Hunyuan3DPaintConfig(max_num_view=6,resolution=512)
config.realesrgan_ckpt_path='hy3dpaint/ckpt/RealESRGAN_x4plus.pth'
config.multiview_cfg_path='hy3dpaint/cfgs/hunyuan-paint-pbr.yaml'
config.custom_pipeline='hy3dpaint/hunyuanpaintpbr'
start=time.monotonic()
pipeline=Hunyuan3DPaintPipeline(config)
pipeline(mesh_path=str(OUT/'cybertruck-shape-hunyuan.glb'),
    image_path=str(BASE/'inputs/references/2026-cybertruck.png'),
    output_mesh_path=str(OUT/'cybertruck-pbr-hunyuan.obj'),
    use_remesh=True,save_glb=True)
output=OUT/'cybertruck-pbr-hunyuan.glb'
assert output.exists(), 'PBR export did not create a GLB'
import hashlib,trimesh
scene=trimesh.load(output,force='scene')
assert scene.geometry,'Empty PBR output'
metadata={'file':str(output),'bytes':output.stat().st_size,
 'triangles':sum(len(g.faces) for g in scene.geometry.values()),
 'sha256':hashlib.sha256(output.read_bytes()).hexdigest(),
 'seconds':round(time.monotonic()-start,2),'accepted':False,
 'generator':'Hunyuan3D-2.1 PBR','views':6,'resolution':512}
(OUT/'cybertruck-pbr-hunyuan.json').write_text(json.dumps(metadata,indent=2))
print(json.dumps(metadata))
'''))
with (QUALITY/'pbr-generation.log').open('w') as log:
    result=subprocess.run([sys.executable,str(worker)],stdout=log,stderr=subprocess.STDOUT,text=True)
print((QUALITY/'pbr-generation.log').read_text()[-5000:])
print('Exit:',result.returncode)
assert result.returncode==0


I0000 00:00:1794899213.447043   17227 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794899213.518292   17227 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794899216.093004   17227 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794899216.093986   17227 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

In [18]:
from pathlib import Path
q=Path('/content/drive/MyDrive/DREV Fleet 2026/quality-lab')
for name in ['pbr-generation.log','rasterizer-build.log','inpaint-build.log']:
 p=q/name
 print('FILE',name, p.read_text()[-4500:] if p.exists() else 'not created')
print('PBR GLB exists:',(q/'cybertruck-pbr-hunyuan.glb').exists())


FILE pbr-generation.log WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1794899213.447043   17227 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794899213.518292   17227 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794899216.093004   17227 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:

In [19]:
import base64,zlib,hashlib,json
from IPython.display import display,HTML
p=QUALITY/'cybertruck-pbr-hunyuan.glb'
raw=p.read_bytes()
encoded=base64.b64encode(zlib.compress(raw,9)).decode()
print(json.dumps({'bytes':len(raw),'encodedBytes':len(encoded),'sha256':hashlib.sha256(raw).hexdigest()}))
display(HTML('<textarea aria-label="Hunyuan PBR model transfer" style="width:95%;height:70px">'+encoded+'</textarea>'))


{"bytes": 4899156, "encodedBytes": 6305872, "sha256": "ca06810dc7fb7a13e93ee8a1a659d975143690077fb96cb6691f17a8c43e948e"}


In [16]:
import sysconfig,shutil
r=Path('/content/Hunyuan3D-2.1/hy3dpaint/DifferentiableRenderer')
source=r/'mesh_inpaint_processor'
assert source.read_bytes()[:4]==b'\x7fELF'
target=r/('mesh_inpaint_processor'+sysconfig.get_config_var('EXT_SUFFIX'))
shutil.copy2(source,target)
probe=subprocess.run([sys.executable,'-c',"import sys;sys.path.insert(0,"+repr(str(r))+");from mesh_inpaint_processor import meshVerticeInpaint;print('Inpainting extension import passed')"],capture_output=True,text=True)
print(target.name,probe.stdout,probe.stderr)
assert probe.returncode==0


mesh_inpaint_processor.cpython-313-x86_64-linux-gnu.so Inpainting extension import passed
 


In [ ]:
basic=Path('/content/basicsr-1.4.2')
p=basic/'setup.py'
s=p.read_text().replace("exec(compile(f.read(), version_file, 'exec'))","namespace = {}\n        exec(compile(f.read(), version_file, 'exec'), namespace)").replace("return locals()['__version__']","return namespace['__version__']")
p.write_text(s)
fixlog=QUALITY/'basicsr-fix.log'
with fixlog.open('w') as log:
 r=subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation',str(basic)],stdout=log,stderr=subprocess.STDOUT,text=True)
print(fixlog.read_text()[-4000:])
assert r.returncode==0


udart,cufft,cufile,cupti,curand,cusolver,cusparse,nvjitlink,nvrtc,nvtx]==13.0.3; platform_system == "Linux"->torch>=1.7->basicsr==1.4.2) (12.6.3.3)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 24.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 139.6 MB/s  0:00:00
  Created wheel for basicsr: filename=basicsr-1.4.2-py3-none-any.whl size=214910 sha256=9937cf21a1cd42e192662fc6a98f948e6d55a9c11cb3569cd82dddc8b1745242
  Stored in directory: /tmp/pip-ephem-wheel-cache-pf3e97si/wheels/db/c2/2f/e098e0af93d7f59d0806da813f92b6dc346767351330b21561
Successfully built basicsr




In [ ]:
import urllib.request,tarfile,json
meta=json.load(urllib.request.urlopen('https://pypi.org/pypi/basicsr/1.4.2/json'))
src=next(r for r in meta['urls'] if r['packagetype']=='sdist')
archive=Path('/content/basicsr-1.4.2.tar.gz')
if not archive.exists(): urllib.request.urlretrieve(src['url'],archive)
import hashlib
assert hashlib.sha256(archive.read_bytes()).hexdigest()==src['digests']['sha256']
with tarfile.open(archive) as tar: tar.extractall('/content',filter='data')
basic=Path('/content/basicsr-1.4.2')
print((basic/'setup.py').read_text()[:6500])
import zipfile,shutil
with zipfile.ZipFile('/content/DREV-Fleet-Colab-Bundle-v5.zip') as z: z.extractall(BASE/'inputs')
for p in (BASE/'inputs/notebooks').glob('*.ipynb'): shutil.copy2(p,BASE/'notebooks'/p.name)
print('Fleet queue and references saved:',len(list((BASE/'inputs/references').glob('*.png'))))


#!/usr/bin/env python

from setuptools import find_packages, setup

import os
import subprocess
import time

version_file = 'basicsr/version.py'


def readme():
    with open('README.md', encoding='utf-8') as f:
        content = f.read()
    return content


def get_git_hash():

    def _minimal_ext_cmd(cmd):
        # construct minimal environment
        env = {}
        for k in ['SYSTEMROOT', 'PATH', 'HOME']:
            v = os.environ.get(k)
            if v is not None:
                env[k] = v
        # LANGUAGE is used on win32
        env['LANGUAGE'] = 'C'
        env['LANG'] = 'C'
        env['LC_ALL'] = 'C'
        out = subprocess.Popen(cmd, stdout=subprocess.PIPE, env=env).communicate()[0]
        return out

    try:
        out = _minimal_ext_cmd(['git', 'rev-parse', 'HEAD'])
        sha = out.strip().decode('ascii')
    except OSError:
        sha = 'unknown'

    return sha


def get_hash():
    if os.path.exists('.git'):
        sha = get_git_hash()[:7]
    # curre

In [ ]:
print(subprocess.run(['nvidia-smi','--query-compute-apps=pid,used_memory','--format=csv'],capture_output=True,text=True).stdout)
print((QUALITY/'shape-generation.log').read_text()[-2200:])
print('Export exists:',(QUALITY/'cybertruck-shape-hunyuan.glb').exists())


pid, used_gpu_memory [MiB]

*args, **kwargs)
  File "/content/Hunyuan3D-2.1/hy3dshape/hy3dshape/models/autoencoders/attention_blocks.py", line 307, in forward
    x = x + self.attn(self.ln_1(x), self.ln_2(data))
            ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1783, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1794, in _call_impl
    return forward_call(*args, **kwargs)
  File "/content/Hunyuan3D-2.1/hy3dshape/hy3dshape/models/autoencoders/attention_blocks.py", line 269, in forward
    x = self.attention(x, data)
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1783, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/to

In [ ]:
source=bake_path.read_text().replace('positions = torch.tensor(positions_texture.reshape(-1, 4)[:, :-1])','positions = torch.tensor(positions_texture.reshape(-1, 4)[:, :-1], device=scene_code.device)').replace('queried_grid["color"].numpy()', 'queried_grid["color"].detach().cpu().numpy()')
bake_path.write_text(source)
with log_path.open('w') as log:
    result=subprocess.run(cmd,cwd='/content/TripoSR',stdout=log,stderr=subprocess.STDOUT,text=True)
print(log_path.read_text()[-4000:])
print('Exit:',result.returncode)
assert result.returncode==0

2026-11-17 05:02:00,847 - INFO - Initializing model ...
2026-11-17 05:02:04,529 - INFO - NumExpr defaulting to 2 threads.
I0000 00:00:1794891725.573713    5455 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794891728.739831    5455 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-17 05:02:17,983 - INFO - Initializing model finished in 17136.76ms.
2026-11-17 05:02:17,985 - INFO - Processing images ...
2026-11-17 05:02:22,462 - INFO - Processing images finished in 4477.25ms.
2026-11-17 05:02:22,462 - INFO - Running image 1/1 ...
2026-11-17 05:02:22,462 - INFO - Running model ...
2026-11-17 05:02:23,658 - INFO - Running model finished in 1195.77ms.
2026-11-17 05:02:23,658 - INFO - Extracting mesh ...
2026-11-17 05:02: